# Scott's CLI Development & Demo Notebook

This notebook demonstrates the CodeCorrect CLI tool for spell-checking code typos using edit distance. It covers:

1. **CLI Integration**: How to use the spell checker from command line
2. **Scaling to 50K+ Vocabulary**: Loading large word lists efficiently
3. **Real-World Testing**: Measuring accuracy on actual typo datasets
4. **Performance Metrics**: Timing and quality analysis

## Overview

**Task**: Integrate CodeCorrect CLI tool, scale to 50K vocabulary, test with real-world typo datasets.

**What this branch delivers**:
- ✅ `src/spell_checker.py` — Production CLI tool
- ✅ DP implementations (naive, memoized, tabulation)
- ✅ Large vocabulary support (~370K real English words or 50K synthetic)
- ✅ Typo dataset with accuracy testing
- ✅ Interactive examples in this notebook

**Key Features**:
- Automatically downloads real-world English dictionary if missing
- Falls back to synthetic vocabulary for quick testing
- Supports 3 DP algorithms (choose by `--method` flag)
- Ranks suggestions by edit distance
- Measures accuracy on real typos

## Import Required Libraries

## Setting Up Large Vocabulary

Before scaling tests, you need a large vocabulary file (~50K+). We use **real-world English dictionaries** instead of synthetic data for realistic spell-checking scenarios.

### Option 1: Real-World English Dictionary (Recommended)

Download from [dwyl/english-words](https://github.com/dwyl/english-words) (~370K words):

```bash
cd data/
curl -O https://raw.githubusercontent.com/dwyl/english-words/master/words_alpha.txt
cd ..
```

**File:** `words_alpha.txt` (~5.8 MB, 367K words)
- Real English words
- Clean (alphabetic only, no symbols/numbers)
- Perfect for realistic accuracy testing
- Public domain / Unlicense

### Option 2: Generate Synthetic Vocabulary (Fallback)

For quick testing without downloading:

```bash
python3 -c "with open('data/large_vocab.txt', 'w') as f: [f.write(f'word{i}\n') for i in range(50000)]"
```

**File:** `large_vocab.txt` (~400 KB, 50K words)
- Generated instantly
- Good for performance testing
- Not linguistically realistic

### Why Not in Git?

Large vocab files (~5+ MB) are excluded from git (see `.gitignore`) to keep the repository lightweight. Download them locally on-demand using the commands above.

**The cells below will:**
1. Auto-download `words_alpha.txt` if missing
2. Fall back to synthetic if download fails
3. Run scaling and accuracy tests

In [1]:
import os
import subprocess
import pandas as pd
import time
import sys
sys.path.insert(0, '../src')

from vocab_loader import load_vocabulary
from tabulation import edit_distance_tabulation

# Ensure data directory exists
os.makedirs('../data', exist_ok=True)

## Integrate into CodeCorrect CLI Tool

In [2]:
# Example CLI call
result = subprocess.run(['python', '../src/spell_checker.py', '--word', 'prin', '--vocab', '../data/python_keywords.txt', '--method', 'tabulation'], capture_output=True, text=True)
print(result.stdout)

Suggestions for 'prin':
  print (distance: 1)
  in (distance: 2)
  True (distance: 3)
  elif (distance: 3)
  from (distance: 3)



## Scale Vocabulary to 50K

In [3]:
# Ensure we have a large vocabulary file
vocab_file = '../data/words_alpha.txt'
if not os.path.exists(vocab_file):
    print("Large vocabulary file not found. Attempting to download...")
    subprocess.run(['curl', '-o', vocab_file, 
                    'https://raw.githubusercontent.com/dwyl/english-words/master/words_alpha.txt'],
                    check=False)
    if os.path.exists(vocab_file):
        print(f"Downloaded: {vocab_file}")
    else:
        print("Download failed. Using synthetic vocabulary instead.")
        vocab_file = '../data/large_vocab.txt'
        if not os.path.exists(vocab_file):
            print(f"Generating synthetic vocabulary to {vocab_file}...")
            with open(vocab_file, 'w') as f:
                for i in range(50000):
                    f.write(f'word{i}\n')
            print("Generated 50K synthetic words.")

# Load large vocabulary
print(f"\nLoading vocabulary from: {vocab_file}")
start = time.time()
vocab = load_vocabulary(vocab_file)
load_time = time.time() - start
print(f"Loaded {len(vocab):,} words in {load_time:.2f} seconds")

# Test edit distance on a subset (for speed)
word = 'test' if len(vocab) > 0 and 'test' in vocab else vocab[100]
subset_size = min(1000, len(vocab))
print(f"\nComputing distances for word '{word}' against {subset_size} vocab words...")
start = time.time()
distances = [(edit_distance_tabulation(word, v), v) for v in vocab[:subset_size]]
dist_time = time.time() - start
print(f"Computed distances in {dist_time:.3f} seconds")
print(f"Top 5 suggestions: {sorted(distances)[:5]}")


Loading vocabulary from: ../data/words_alpha.txt
Loaded 370,105 words in 0.14 seconds

Computing distances for word 'test' against 1000 vocab words...
Computed distances in 0.009 seconds
Top 5 suggestions: [(3, 'aas'), (3, 'abbest'), (3, 'abdest'), (3, 'abet'), (3, 'abit')]


## Test with Real-World Typo Datasets

In [4]:
# Load typo dataset
typo_file = '../data/typo_dataset.csv'
if not os.path.exists(typo_file):
    print(f"Note: Typo dataset not found at {typo_file}")
    print("Creating a sample typo dataset for testing...")
    sample_typos = {
        'typo': ['prin', 'defn', 'improt', 'whiel', 'retrun', 'fro'],
        'correct': ['print', 'def', 'import', 'while', 'return', 'for']
    }
    typos = pd.DataFrame(sample_typos)
    typos.to_csv(typo_file, index=False)
    print(f"Created sample dataset: {typo_file}")
else:
    typos = pd.read_csv(typo_file)

print(f"\nTypo Dataset ({len(typos)} entries):")
print(typos.head(10))

# Use a reasonable vocabulary for accuracy testing
test_vocab_file = '../data/python_keywords.txt'
if os.path.exists(test_vocab_file):
    test_vocab = load_vocabulary(test_vocab_file)
    print(f"\nUsing Python keywords vocabulary ({len(test_vocab)} words)")
else:
    print("\nPython keywords file not found. Extracting from typos...")
    test_vocab = list(set(typos['correct'].tolist()))
    print(f"Using {len(test_vocab)} unique correct words from dataset")

# Test accuracy
print("\n=== Accuracy Test ===")
correct = 0
total = len(typos)
for idx, row in typos.iterrows():
    typo = row['typo']
    expected = row['correct']
    # Find best match
    suggestions = sorted([(edit_distance_tabulation(typo, v), v) for v in test_vocab])
    best = suggestions[0][1] if suggestions else None
    is_correct = (best == expected)
    correct += is_correct
    status = "✓" if is_correct else "✗"
    print(f"{status} '{typo}' → best: '{best}', expected: '{expected}'")

accuracy = correct / total * 100
print(f"\nFinal Accuracy: {accuracy:.1f}% ({correct}/{total})")


Typo Dataset (10 entries):
     typo correct
0    prin   print
1    defn     def
2  improt  import
3   flase   False
4    ture    True
5  retrun  return
6   whiel   while
7     fro     for
8     els    else
9    clas   class

Using Python keywords vocabulary (36 words)

=== Accuracy Test ===
✓ 'prin' → best: 'print', expected: 'print'
✓ 'defn' → best: 'def', expected: 'def'
✓ 'improt' → best: 'import', expected: 'import'
✗ 'flase' → best: 'class', expected: 'False'
✗ 'ture' → best: 'try', expected: 'True'
✓ 'retrun' → best: 'return', expected: 'return'
✓ 'whiel' → best: 'while', expected: 'while'
✗ 'fro' → best: 'from', expected: 'for'
✓ 'els' → best: 'else', expected: 'else'
✓ 'clas' → best: 'class', expected: 'class'

Final Accuracy: 70.0% (7/10)
